# Day3_12 · 여러 조건을 함께 보는 회귀

콘크리트 배합과 재령을 함께 사용해 압축강도를 설명하고 예측합니다.

## 이 노트북에서 확인할 내용

- 여러 입력열로 하나의 회귀식을 만들 수 있습니다.
- 예측값·평가지표·회귀계수를 서로 다른 결과로 읽을 수 있습니다.
- 계수의 단위와 신뢰구간을 함께 확인할 수 있습니다.

## 실행 방법

위에서 아래로 코드 셀을 한 개씩 실행하고 바로 다음 결과를 확인합니다. 수정 안내가 있는 곳에서는 표시된 값 하나만 바꾼 뒤 결과를 비교합니다.

### D3B-12-01 · 경로 도구 불러오기

In [ ]:
from pathlib import Path

### D3B-12-02 · 현재 폴더 확인

In [ ]:
current_folder = Path.cwd()

In [ ]:
current_folder

### D3B-12-03 · 데이터 폴더 후보 만들기

In [ ]:
data_candidates = [
    current_folder / 'data',
    current_folder.parent / 'data',
]

In [ ]:
data_candidates

### D3B-12-04 · 데이터 폴더 선택

In [ ]:
DATA_DIR = next(folder for folder in data_candidates if folder.exists())

In [ ]:
DATA_DIR

### D3B-12-05 · pandas와 numpy 불러오기

In [ ]:
import pandas as pd
import numpy as np

### D3B-12-06 · 그림 도구 불러오기

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

### D3B-12-07 · 콘크리트 강도자료 읽기

In [ ]:
concrete = pd.read_csv(DATA_DIR / 'concrete_strength.csv')

In [ ]:
concrete.head()

### D3B-12-08 · 입력 열 목록 정하기

In [ ]:
feature_columns = ['cement_kg_m3', 'slag_kg_m3', 'fly_ash_kg_m3', 'water_kg_m3', 'superplasticizer_kg_m3', 'coarse_aggregate_kg_m3', 'fine_aggregate_kg_m3', 'age_days']

In [ ]:
pd.Series(feature_columns, name='input_column')

### D3B-12-09 · 입력표 만들기

In [ ]:
X = concrete[feature_columns]

In [ ]:
X.head()

### D3B-12-10 · 결과 열 만들기

In [ ]:
y = concrete['strength_mpa']

In [ ]:
y.head()

## 여러 입력을 동시에 사용하는 회귀

한 계수는 나머지 입력값이 같은 조건에서 해당 입력이 한 단위 달라질 때 예측값이 얼마나 달라지는지를 나타냅니다. 서로 단위가 다르므로 계수의 절댓값만으로 중요도를 비교하지 않습니다.

### D3B-12-11 · 자료 분할 함수 불러오기

In [ ]:
from sklearn.model_selection import train_test_split

### D3B-12-12 · 학습자료와 평가자료 나누기

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

In [ ]:
pd.Series({'train_rows': len(X_train), 'test_rows': len(X_test)})

### D3B-12-13 · 선형회귀 도구 불러오기

In [ ]:
from sklearn.linear_model import LinearRegression

### D3B-12-14 · 여러 입력으로 회귀식 맞추기

In [ ]:
multiple_model = LinearRegression().fit(X_train, y_train)

In [ ]:
multiple_model

### D3B-12-15 · 평가자료 예측하기

In [ ]:
multiple_prediction = multiple_model.predict(X_test)

In [ ]:
pd.Series(multiple_prediction[:5], name='predicted_strength_mpa')

### D3B-12-16 · 실제값과 예측값 비교

In [ ]:
prediction_table = pd.DataFrame({'actual_mpa': y_test.to_numpy(), 'predicted_mpa': multiple_prediction})

In [ ]:
prediction_table.head(10).round(2)

### D3B-12-17 · 회귀 평가지표 도구 불러오기

In [ ]:
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score

### D3B-12-18 · 평가 결과표 만들기

In [ ]:
multiple_metrics = pd.Series({
    'MAE': mean_absolute_error(y_test, multiple_prediction),
    'RMSE': root_mean_squared_error(y_test, multiple_prediction),
    'R2': r2_score(y_test, multiple_prediction),
})

In [ ]:
multiple_metrics.round(3)

### D3B-12-19 · 회귀계수 표 만들기

In [ ]:
coefficient_table = pd.DataFrame({'input_column': feature_columns, 'coefficient': multiple_model.coef_})

In [ ]:
coefficient_table.round(4)

### D3B-12-20 · 계수를 가로 막대로 보기

In [ ]:
coefficient_table.sort_values('coefficient').plot.barh(x='input_column', y='coefficient', legend=False, figsize=(7, 5))
plt.xlabel('Change in predicted MPa per one input unit')
plt.show()

계수는 각 열의 단위에 의존합니다. 예를 들어 재령 1일과 시멘트 1 kg/m³는 같은 크기의 변화가 아니므로 막대 길이를 그대로 중요도 순위로 사용하지 않습니다.

### D3B-12-21 · 통계적 회귀 도구 불러오기

In [ ]:
import statsmodels.api as sm

### D3B-12-22 · 상수항이 있는 입력표 만들기

In [ ]:
X_with_constant = sm.add_constant(X)

In [ ]:
X_with_constant.head()

### D3B-12-23 · 전체 자료로 회귀 결과 계산

In [ ]:
inference_model = sm.OLS(y, X_with_constant).fit()

In [ ]:
inference_model

### D3B-12-24 · 계수의 95% 구간 가져오기

In [ ]:
coefficient_intervals = inference_model.conf_int()

In [ ]:
coefficient_intervals.head().round(4)

### D3B-12-25 · 계수와 구간을 한 표로 만들기

In [ ]:
inference_table = pd.DataFrame({
    'coefficient': inference_model.params,
    'lower_95': coefficient_intervals[0],
    'upper_95': coefficient_intervals[1],
})

In [ ]:
inference_table.round(4)

이 구간은 선형성, 오차 구조와 관측의 독립성 같은 회귀 가정 아래 계산됩니다. 다음 교시에서 잔차와 입력변수 관계를 확인한 뒤 해석 범위를 정합니다.

### D3B-12-26 · 확인할 입력 열 줄이기

In [ ]:
# 목록에서 열 하나를 빼거나 다시 넣어 보세요.
selected_features = ['cement_kg_m3', 'water_kg_m3', 'age_days']

In [ ]:
pd.Series(selected_features)

### D3B-12-27 · 선택한 열로 회귀식 맞추기

In [ ]:
selected_model = LinearRegression().fit(X_train[selected_features], y_train)

In [ ]:
selected_model

### D3B-12-28 · 선택한 회귀식 평가하기

In [ ]:
selected_prediction = selected_model.predict(X_test[selected_features])
selected_metrics = pd.Series({
    'MAE': mean_absolute_error(y_test, selected_prediction),
    'RMSE': root_mean_squared_error(y_test, selected_prediction),
    'R2': r2_score(y_test, selected_prediction),
})

In [ ]:
selected_metrics.round(3)

### D3B-12-CODEX · Codex에 확장 분석 요청하기

In [ ]:
codex_request = '''concrete DataFrame과 feature_columns를 사용하세요. 기존 셀은 수정하지 마세요. 다중선형회귀의 coefficient, 95% confidence interval을 정리한 표를 새 셀로 제안해 주세요. 단위가 다른 계수의 절댓값을 중요도처럼 비교하지 말라는 주의 문장을 포함해 주세요.'''.strip()

In [ ]:
codex_request

### D3B-12-CHECK · 제안받은 코드를 실행하기 전에 확인하기

In [ ]:
codex_checklist = pd.Series(['결과 열 strength_mpa가 입력에 들어가지 않는가?', '계수와 구간의 열 이름이 분명한가?', '계수를 인과효과나 중요도라고 단정하지 않는가?'], name='확인할 내용')

In [ ]:
codex_checklist